# Step 02 — probes to genes

**Data type: RNA_array** (GSE65391). **Reads:** the series matrix, `GPL10558.soft.gz`, NCBI gene_info.
**Writes:** `step02_expression.rds`.

The array measures about 48,000 probes, and several probes can target one gene. The platform
annotation gives each probe an NCBI Entrez gene identifier. Control probes have none and are dropped.

**One probe per gene: the brightest.** The probe with the highest mean intensity sits furthest above
background, so it measures the gene with the least noise.

**Genes are keyed by Entrez identifier and named with the current NCBI symbol.** The RNA sequencing
studies in steps 22–25 reach the same identifiers, so every study uses the same gene names. The array
annotation is from 2010 and some of its symbols have since changed (for example EPB49 is now DMTN).

In [1]:
source("../src/paths.R")
source("../src/platforms.R")
suppressMessages({library(GEOquery); library(Biobase)})
X    <- exprs(suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE)))
gpl  <- Table(suppressMessages(getGEO(filename = raw("GSE65391", "GPL10558.soft.gz"))))
gi   <- read_gene_info()
meta <- readRDS(art("step01_metadata.rds"))
stopifnot(identical(colnames(X), rownames(meta)))
c(probes = nrow(X), samples = ncol(X), min = round(min(X, na.rm = TRUE), 2), max = round(max(X, na.rm = TRUE), 2))

probes  samples      min      max 
43799.00   996.00     1.92    15.65

In [2]:
entrez <- gpl$Entrez_Gene_ID[match(rownames(X), gpl$ID)]
entrez[!is.na(entrez) & !nzchar(entrez)] <- NA
ok <- !is.na(entrez) & rowSums(is.na(X)) == 0
E  <- collapse_max_mean(X[ok, ], entrez[ok])          # rows are Entrez identifiers
symbol <- gi$Symbol[match(rownames(E), gi$GeneID)]
# an identifier retired since 2010 has no current symbol; keep the array's symbol
old <- gpl$Symbol[match(rownames(E), gpl$Entrez_Gene_ID)]
symbol[is.na(symbol)] <- old[is.na(symbol)]
genes <- data.frame(entrez = rownames(E), symbol = symbol, stringsAsFactors = FALSE)
genes <- genes[!duplicated(genes$symbol), ]
E <- E[genes$entrez, ]; rownames(E) <- genes$symbol
c(probes_with_entrez = sum(ok), genes = nrow(E), renamed_since_2010 = sum(symbol != old, na.rm = TRUE))

probes_with_entrez              genes renamed_since_2010 
             40530              28948               5977

## Mark genes expressed in whole blood

Genes not expressed in blood sit at the floor of this array's scale, log2(10) = 3.32. A gene counts as
expressed when its log2 intensity is above 6 in at least 10% of samples. The 10% rule keeps genes that
are switched on in only some patients, which is the kind of gene an endotype may depend on.
Clustering uses expressed genes only.

**The interferon score does not use this filter.** SIGLEC1 is a strong interferon marker, but on this
array it is at the floor in most samples. Filtering first would silently remove it from the score.

In [3]:
expressed <- rowMeans(E > 6) >= 0.10
ifn6 <- read_gene_set("ifn-type1-6.txt")
stopifnot(all(ifn6 %in% rownames(E)))
rbind(expressed = expressed[ifn6])
c(genes = nrow(E), expressed = sum(expressed))

,IFI27,IFI44L,IFIT1,ISG15,RSAD2,SIGLEC1
expressed,TRUE,TRUE,TRUE,TRUE,TRUE,FALSE


genes expressed 
    28948      8825

In [4]:
saveRDS(list(E = E, expressed = expressed, genes = genes, meta = meta), art("step02_expression.rds"))
cat("wrote", art("step02_expression.rds"), "\n")

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step02_expression.rds 


## Findings

40,530 annotated probes collapse to 28,948 genes, and 5,977 of them carry a newer NCBI symbol than the
array annotation gives. 8,825 genes are expressed in whole blood. Every interferon-score gene is on
the array. SIGLEC1 is present but below the expression threshold, which is why the score works from
the full gene table.